# `ai_parse_document` — Old Article Preprocessing & Extraction

## Context
The notebook `ai_parse_document_debug` showed that Databricks `ai_parse_document` returns
**0 elements** on `old_articles.pdf` — a 1908 French journal scanned at very low resolution
with misalignment, noise, and tilt.

![Article preview](https://raw.githubusercontent.com/O-Faraday/databricks_genai_demo/main/data/multiformat/old_article_presentation.png)

## Approach
Rather than accepting the failure, this notebook applies a **preprocessing pipeline** before
re-parsing. Three classical image-processing techniques are chained per page:

| Step | Technique | Problem solved |
|---|---|---|
| **Sharpen** | Unsharp mask (PIL) | Recovers edge detail lost in scan |
| **Binarise** | Otsu global threshold (numpy) | Removes grey noise → sharp B&W |
| **Deskew** | Projection-profile angle scan | Corrects tilt from scanner bed |

The preprocessed pages are reassembled into a new PDF and re-submitted to `ai_parse_document`.

## Notebook structure
| Section | Description |
|---|---|
| **1. Set up** | Paths, file registry |
| **2. Parse function** | `ai_parse_document` v2.0 SQL wrapper |
| **3. Preprocessing pipeline** | Sharpen → Binarise → Deskew → rebuild PDF |
| **3b. Re-parse & visualise** | Compare baseline vs preprocessed extraction |



In [0]:
# Install required libraries.
# - pymupdf  (fitz)  : PDF rasterisation at target DPI — no poppler binary required
# - Pillow           : image manipulation (sharpen, rotate, binarise)
# - numpy            : Otsu thresholding and projection-profile deskew
%pip install -q pymupdf Pillow numpy --quiet
# PyMuPDF (fitz) replaces pdf2image: pure Python, no poppler binary required
dbutils.library.restartPython()


### Config Unity Catalog

Load the shared `catalog` and `schema` variables from the central configuration notebook.

In [0]:
%run ../_config/config_unity_catalog

### Load the debug visualisation function

Imports `render_ai_parse_output_interactive()` from the companion `document_renderer` notebook.
This function renders `ai_parse_document` output as an interactive bounding-box overlay on page images.

In [0]:
%run ./document_renderer

## 1. Set up

Define Unity Catalog volume paths and the file registry. Add entries to `FILES` to process additional documents.

In [0]:
import json
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# ── Paths ──────────────────────────────────────────────────────────────────
PATH_VOLUME   = f"/Volumes/{catalog}/{schema}/raw_data/multiformat"
PATH_RESULTS  = f"/Volumes/{catalog}/{schema}/raw_data/parse_results"

# Create output directories if they don't exist
dbutils.fs.mkdirs(PATH_RESULTS)

# ── Files to analyse ───────────────────────────────────────────────────────
FILES = {
    "old_articles"  : "old_articles.pdf",
}

print(f"Volume path   : {PATH_VOLUME}")
print(f"Results output: {PATH_RESULTS}")
print(f"Files to parse: {list(FILES.values())}")

## 2. Parse Document Function with `ai_parse_document`

Wraps the Databricks `ai_parse_document` SQL function into a reusable Python helper.

Key parameters used:
- `version = '2.0'` — latest schema (Jan 2026); output is an `elements` array (replaces the legacy `pages` array from v1.0)
- `descriptionElementTypes = 'figure'` — in v2.0, only figures receive AI-generated text descriptions; passing `'*'` produces the same behaviour
- `imageOutputPath` — persists rendered page images to the UC volume, enabling visual inspection and potential multimodal RAG pipelines

The function reads the file as binary via `read_files(..., format => 'binaryFile')` and returns a list of parsed result objects, one per input file.

In [0]:
def parse_doc(source, output):
    """Parse a document with ai_parse_document v2.0 and return the results.

    Parameters
    ----------
    source : str
        Full UC volume path to the source file (PDF, image, PPTX, …).
    output : str
        UC volume path where page images will be written by ai_parse_document.

    Returns
    -------
    list
        One parsed result struct per input file, containing an `elements` array
        with type, content, bounding box, confidence, and page reference.
    """
    # read_files with format='binaryFile' streams the raw bytes to ai_parse_document.
    # The CTE isolates the parsing step so the outer SELECT can filter or join freely.
    sql = f'''
    WITH parsed_documents AS (
        SELECT
            path,
            ai_parse_document(
                content,
                map(
                    'version',                '2.0',
                    'imageOutputPath',        '{output}',
                    'descriptionElementTypes','figure'
                )
            ) AS parsed
        FROM read_files('{source}', format => 'binaryFile')
    )
    SELECT * FROM parsed_documents
    '''

    # .collect() materialises all rows on the driver — acceptable for single-document
    # calls; for bulk processing prefer writing to Delta and reading back.
    parsed_results = [row.parsed for row in spark.sql(sql).collect()]
    return parsed_results

## 3. Preprocess Scanned Document Deep Dive — `old_articles.pdf`

The baseline parse returned **0 elements** because `old_articles.pdf` is a 1908 French journal scanned at very low resolution (~72 dpi equivalent). This section applies a three-step image processing pipeline to rescue the document before re-submitting it to `ai_parse_document`.

### Pipeline overview

| Step | Technique | Purpose |
|---|---|---|
| **1. Sharpen** | Unsharp mask — radius 1.5, 120 % | Restores edge detail lost in scan compression |
| **2. Binarise** | Otsu global threshold (pure numpy) | Converts grey scan to crisp B&W, eliminates noise |
| **3. Deskew** | Projection-profile angle sweep ±5° | Corrects rotational tilt from scanner bed |

The processed pages are reassembled into a new PDF (`preprocessed_old_articles.pdf`) at 300 DPI and saved back to the UC volume.

In [0]:
import io, time
import numpy as np
import fitz                              # PyMuPDF — no poppler required
from PIL import Image, ImageFilter

SOURCE_PDF       = f"{PATH_VOLUME}/{FILES['old_articles']}"
PREPROCESSED_PDF = f"{PATH_VOLUME}/preprocessed_{FILES['old_articles']}"
TARGET_DPI       = 300
ZOOM             = TARGET_DPI / 72.0     # fitz renders at 72 dpi by default

pdf_doc = fitz.open(SOURCE_PDF)
print(f"Source PDF : {SOURCE_PDF}")
print(f"Pages      : {pdf_doc.page_count}  |  target DPI: {TARGET_DPI}")


def deskew(pil_img, angle_range=5.0, steps=100):
    """Correct rotational skew using the projection-profile method.

    The image is rotated at `steps` angles within ±angle_range degrees.
    The angle that maximises the variance of horizontal row-pixel sums is chosen:
    a well-aligned text page produces sharp peaks and troughs in the row histogram,
    yielding high variance; a tilted page smears those peaks, yielding low variance.

    Skips correction if the best angle is below 0.1° (imperceptible tilt).
    """
    gray = np.array(pil_img.convert('L'))
    best_angle, best_score = 0.0, -1.0
    for angle in np.linspace(-angle_range, angle_range, steps):
        rotated  = Image.fromarray(gray).rotate(angle, expand=False, fillcolor=255)
        row_sums = np.array(rotated).sum(axis=1).astype(float)
        score    = float(np.var(row_sums))
        if score > best_score:
            best_score, best_angle = score, angle
    if abs(best_angle) > 0.1:
        return pil_img.rotate(best_angle, expand=False, fillcolor=255,
                               resample=Image.BICUBIC)
    return pil_img


def binarise(pil_img):
    """Binarise a greyscale image using Otsu's global threshold (pure numpy).

    Otsu's method finds the threshold t* that minimises intra-class pixel variance
    (equivalently, maximises inter-class variance between background and foreground).
    This is more robust than a fixed threshold for scanned documents where the
    background brightness varies across pages or within a page.
    """
    gray    = np.array(pil_img.convert('L'))
    hist, _ = np.histogram(gray.flatten(), 256, [0, 256])
    hist    = hist.astype(float)
    total   = gray.size
    sum_all = float((np.arange(256) * hist).sum())
    wb = sb  = 0.0
    best_thresh, best_var = 0, 0.0
    for t in range(256):
        wb += hist[t]; wf = total - wb
        if wb == 0 or wf == 0: continue
        sb += t * hist[t]
        mb, mf = sb / wb, (sum_all - sb) / wf
        var = wb * wf * (mb - mf) ** 2
        if var > best_var:
            best_var, best_thresh = var, t
    return Image.fromarray((gray > best_thresh).astype('uint8') * 255).convert('RGB')


# ── Rasterise + preprocess each page with fitz ─────────────────────────────
processed_pages = []
mat = fitz.Matrix(ZOOM, ZOOM)            # scale matrix for target DPI

for page_num in range(pdf_doc.page_count):
    t0   = time.time()
    page = pdf_doc.load_page(page_num)
    pix  = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)

    # fitz pixmap -> PIL Image
    raw_img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
    w0, h0  = raw_img.size

    # Step 1: sharpen to recover edge detail lost in scan
    sharpened = raw_img.filter(ImageFilter.UnsharpMask(radius=1.5, percent=120, threshold=3))
    # Step 2: binarise (Otsu)
    binary    = binarise(sharpened)
    # Step 3: deskew
    deskewed  = deskew(binary)

    processed_pages.append(deskewed)
    print(f"  Page {page_num+1}: {w0}x{h0}px -> preprocessed in {round(time.time()-t0,2)}s")

pdf_doc.close()

# ── Re-assemble into PDF ────────────────────────────────────────────────────
processed_pages[0].save(
    PREPROCESSED_PDF, save_all=True,
    append_images=processed_pages[1:],
    format='PDF', resolution=TARGET_DPI,
)
import os
print(f"\nPreprocessed PDF saved : {PREPROCESSED_PDF}")
print(f"Size                   : {os.path.getsize(PREPROCESSED_PDF)//1024} KB")
print(f"Pages                  : {len(processed_pages)}")


### 3b. Re-parse with `ai_parse_document` & visualise

Submit the preprocessed PDF to `ai_parse_document v2.0` and render the results using `render_ai_parse_output_interactive`.

Expected improvement over the raw baseline:
- Element count > 0 (vs 0 on the raw scan)
- Text blocks correctly segmented despite the historical typeface
- Bounding boxes aligned with the upscaled, deskewed page image

In [0]:
parsed_results = parse_doc(PREPROCESSED_PDF, PATH_RESULTS)
render_ai_parse_output_interactive(parsed_results)
